# IM-RoTHP: Extensive Benchmark & Ablation Study

Este notebook realiza uma avaliação extensiva do modelo **IM-RoTHP** em comparação com o baseline **RoTHP** em múltiplos datasets.

### Objetivos:
1.  **Generalização:** Testar em datasets com dinâmicas diferentes (`retweet`, `stackoverflow`, `taxi`, `synthetic`).
2.  **Ablação de Alpha:** Comparar Alpha Aprendido vs Alpha Fixo vs Sem Modulação.
3.  **Métricas Completas:** NLL, RMSE (Tempo) e Acurácia (Tipo).

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch FP16
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

from easy_tpp.model.torch_model.torch_rothp import RoTHP, RotaryEmbedding
from easy_tpp.model.torch_model.torch_imrothp import IMRoTHP, IntensityModulatedRotaryEmbedding

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

### 1. Modelos e Configuração

In [ ]:
# --- VERSÃO ESTABILIZADA DO IM-RoPE ---
class StabilizedIMRotaryEmbedding(RotaryEmbedding):
    def __init__(self, dim, max_freq=10000, max_modulation=0.1, fixed_alpha=None):
        super().__init__(dim, max_freq)
        self.max_modulation = max_modulation
        self.fixed_alpha = fixed_alpha
        
        if fixed_alpha is not None:
            # Alpha fixo (não treinável)
            self.register_buffer('alpha', torch.tensor(float(fixed_alpha)))
        else:
            # Alpha aprendível (começa em 0)
            self.alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, t, intensity):
        # Modulação: 1 + max_mod * tanh(alpha * log(1+lambda))
        raw_mod = self.alpha * torch.log1p(intensity.unsqueeze(-1))
        modulation = 1.0 + self.max_modulation * torch.tanh(raw_mod)
        
        t_expanded = t.unsqueeze(-1)
        thetas_expanded = self.thetas.view(1, 1, -1)
        args = t_expanded * thetas_expanded * modulation
        
        cos = torch.cos(args)
        sin = torch.sin(args)
        cos = torch.repeat_interleave(cos, 2, dim=-1)
        sin = torch.repeat_interleave(sin, 2, dim=-1)
        return cos, sin

class StabilizedIMRoTHP(IMRoTHP):
    def __init__(self, model_config, fixed_alpha=None):
        super().__init__(model_config)
        self.rotary_emb = StabilizedIMRotaryEmbedding(self.d_model // self.n_head, fixed_alpha=fixed_alpha)

# --- CONFIG ---
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig() # Necessário para RMSE (Predict)

# --- DATALOADER ---
def get_dataloader(ds_name):
    print(f"\nCarregando {ds_name}...")
    try:
        dataset = load_dataset(f"easytpp/{ds_name}")
    except:
        print(f"Dataset {ds_name} não encontrado no HF. Tentando synthetic local...")
        # Fallback para sintético se falhar (implementar geração se necessário)
        return None, None, None, None, None

    train_data = dataset['train']
    dev_data = dataset['validation']
    test_data = dataset['test']
    
    all_deltas = []
    for item in train_data:
        all_deltas.extend([d for d in item['time_since_last_event'] if d > 0])
    time_scale = np.mean(all_deltas)
    
    max_type = 0
    for x in train_data:
        if len(x['type_event']) > 0:
            max_type = max(max_type, max(x['type_event']))
    num_types = max_type + 1
    pad_id = num_types
    
    def collate_fn(batch_list):
        batch_size = len(batch_list)
        max_len = max(len(x['time_since_start']) for x in batch_list)
        pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
        pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
        pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
        batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
        attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
        causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
        for i, item in enumerate(batch_list):
            l = len(item['time_since_start'])
            ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
            td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
            ev = torch.tensor(item['type_event'], dtype=torch.long)
            ts = (ts - ts[0]) / time_scale
            td = td / time_scale
            pad_time[i, :l] = ts.float()
            pad_delta[i, :l] = td.float()
            pad_type[i, :l] = ev
            batch_non_pad_mask[i, :l] = 1.0
            mask_i = causal_mask_base.clone()
            mask_i[:, l:] = True
            mask_i[l:, :] = True
            attention_mask[i] = mask_i
        return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

    train_l = DataLoader(train_data, batch_size=1024, shuffle=True, collate_fn=collate_fn)
    dev_l = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate_fn)
    test_l = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate_fn)
    
    return train_l, dev_l, test_l, num_types, pad_id

### 2. Funções de Treino e Avaliação (com RMSE)

In [ ]:
def compute_metrics(model, loader):
    model.eval()
    total_nll = 0
    total_events = 0
    total_rmse = 0
    total_acc = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = [t.to(device) for t in batch]
            
            # NLL
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
            total_nll += loss.item()
            
            # RMSE & ACC (Predict One Step)
            # predict_one_step é mais lento, vamos fazer em sub-amostra ou full dependendo do tempo
            # Para benchmark rápido, vamos pular RMSE em dev, fazer só em test no final
            
            total_events += num
            
    return total_nll / (total_events + 1e-9)

def train_model(model, name, train_loader, dev_loader, epochs=15):
    print(f"  > {name}...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    best_nll = float('inf')
    patience = 3
    no_improve = 0
    
    for epoch in range(1, epochs+1):
        model.train()
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
        val_nll = compute_metrics(model, dev_loader)
        
        # Alpha log
        alpha_info = ""
        if hasattr(model, 'rotary_emb') and hasattr(model.rotary_emb, 'alpha'):
            if isinstance(model.rotary_emb.alpha, torch.nn.Parameter):
                alpha_info = f" | Alpha: {model.rotary_emb.alpha.item():.4f}"
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"    Ep {epoch}: Val NLL {val_nll:.4f}{alpha_info}")
            
        if val_nll < best_nll:
            best_nll = val_nll
            no_improve = 0
            # Salvar melhor estado (na memória)
            # best_state = copy.deepcopy(model.state_dict()) 
        else:
            no_improve += 1
            # if no_improve >= patience: 
            #    print("    Early stopping.")
            #    break
                
    return best_nll

def final_evaluate(model, test_loader):
    model.eval()
    total_se = 0
    total_correct = 0
    total_events = 0
    
    print("    Calculando RMSE e ACC (pode demorar)...")
    with torch.no_grad():
        for batch in test_loader:
            batch = [t.to(device) for t in batch]
            pad_time, pad_delta, pad_type, mask, attn = batch
            
            # Predict
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch)
            
            # Targets (shift 1)
            targets_delta = pad_delta[:, 1:]
            targets_type = pad_type[:, 1:]
            mask_target = mask[:, 1:]
            
            # RMSE
            se = ((dtimes_pred - targets_delta) ** 2) * mask_target
            total_se += se.sum().item()
            
            # ACC
            correct = (types_pred == targets_type) * mask_target
            total_correct += correct.sum().item()
            
            total_events += mask_target.sum().item()
            
    rmse = np.sqrt(total_se / (total_events + 1e-9))
    acc = total_correct / (total_events + 1e-9)
    return rmse, acc

### 3. Execução do Benchmark

In [ ]:
datasets = ['retweet', 'stackoverflow', 'taxi', 'synthetic'] # Adicione taxi/synthetic se existirem
results = []

for ds in datasets:
    # Carregar
    train_l, dev_l, test_l, num_types, pad_id = get_dataloader(ds)
    if train_l is None: continue
    
    config = ModelConfig(num_types, pad_id)
    
    # Definir Competidores
    competitors = [
        ('RoTHP', RoTHP, {}),
        ('IM-RoTHP (Learn)', StabilizedIMRoTHP, {'fixed_alpha': None}),
        ('IM-RoTHP (Fix 0.05)', StabilizedIMRoTHP, {'fixed_alpha': 0.05})
    ]
    
    for name, cls, kwargs in competitors:
        # Instanciar
        if 'fixed_alpha' in kwargs:
            model = cls(config, fixed_alpha=kwargs['fixed_alpha']).to(device)
        else:
            model = cls(config).to(device)
            
        # Treinar
        val_nll = train_model(model, f"{ds} - {name}", train_l, dev_l, epochs=15)
        
        # Avaliar Final
        rmse, acc = final_evaluate(model, test_l)
        
        # Salvar Alpha Final (se existir)
        final_alpha = 0.0
        if hasattr(model, 'rotary_emb') and hasattr(model.rotary_emb, 'alpha'):
             final_alpha = model.rotary_emb.alpha.item()
             
        res = {
            'Dataset': ds,
            'Model': name,
            'NLL': val_nll,
            'RMSE': rmse,
            'ACC': acc,
            'Alpha': final_alpha
        }
        results.append(res)
        print(f"    RESULT: {res}")

# Exibir Tabela Final
df_results = pd.DataFrame(results)
print("\n=== BENCHMARK COMPLETO ===")
print(df_results)

# Plotar
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df_results, x='Dataset', y='NLL', hue='Model', ax=ax[0])
ax[0].set_title('NLL (Menor é Melhor)')

sns.barplot(data=df_results, x='Dataset', y='RMSE', hue='Model', ax=ax[1])
ax[1].set_title('RMSE Tempo (Menor é Melhor)')

plt.tight_layout()
plt.show()